# AICE 에러 해결 가이드 (Troubleshooting)

> 시험 중 흔하게 마주치는 에러 메시지와 그 원인, 해결 방법을 정리했습니다.

### 🚨 ValueError: could not convert string to float
- **원인:** 모델 학습(`fit`)이나 스케일링을 할 때 문자열 컬럼이 포함되어 발생합니다.
- **해결:** 범주형(문자열) 컬럼을 삭제하거나 `get_dummies`로 인코딩하세요.
```python
# 에러 유발 컬럼 확인
print(X_train.dtypes)
# 인코딩 처리
df = pd.get_dummies(df, drop_first=True)
```

### 🚨 ValueError: Found array with dim 3. Estimator expected <= 2.
- **원인:** Scikit-learn 모델에 3차원 이상의 배열을 넣었을 때 발생합니다. (주로 이미지 데이터)
- **해결:** `reshape()`를 사용하여 2차원으로 펼쳐주세요.
```python
X_train_flat = X_train.reshape(X_train.shape[0], -1)
```

### 🚨 ValueError: Input contains NaN, infinity or a value too large for dtype('float64').
- **원인:** 데이터에 결측치(NaN)가 아직 남아있는데 모델 학습을 시도했을 때 발생합니다.
- **해결:** 결측치를 완전히 제거(`dropna`)하거나 적절한 값으로 채우세요(`fillna`).
```python
print(df.isnull().sum()) # 결측치가 남은 컬럼 확인
df = df.fillna(df.mean()) # 평균값으로 채우기
```

### 🚨 AttributeError: 'Series' object has no attribute 'flatten'
- **원인:** pandas Series 객체에는 numpy 배열용 메서드인 flatten이 없습니다.
- **해결:** Series를 numpy 배열로 바꾼 뒤 flatten을 적용하세요.
```python
# 잘못된 방법
# y_test.flatten()
# 올바른 방법
y_test.values.flatten()
```

### 🚨 Shape mismatch (스케일링 관련 에러)
- **원인:** Train과 Test의 컬럼 수가 다를 때 (예: Test 데이터에 특정 범주가 없어서 get_dummies 결과 컬럼 수가 달라짐) 발생합니다.
- **해결:** get_dummies를 할 때 Train과 Test를 합쳐서(concat) 한 번에 인코딩한 후 다시 분리하세요.
```python
# 데이터를 분할하기 전에 인코딩을 먼저 수행하는 것이 안전합니다.
df = pd.get_dummies(df, drop_first=True)
X = df.drop(columns=['target'])
y = df['target']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
```

### 🚨 UnicodeDecodeError: 'cp949' codec can't decode byte / 'utf-8' codec can't decode byte
- **원인:** CSV 파일이 만들어질 때 쓰인 인코딩과 `pd.read_csv()`가 기본으로 시도하는 인코딩이 달라서 발생합니다 (한글이 섞인 데이터에서 흔함).
- **해결:** `encoding` 옵션을 바꿔가며 다시 읽어보세요.
```python
# utf-8로 실패하면 cp949(한글 윈도우 기본 인코딩)로 시도
df = pd.read_csv('data.csv', encoding='cp949')
# 그래도 안 되면 반대로 utf-8-sig로 시도
df = pd.read_csv('data.csv', encoding='utf-8-sig')
```

### 🚨 KeyError: '컬럼명'
- **원인:** 존재하지 않는 컬럼명을 참조했을 때 발생합니다. 오타이거나, `get_dummies`/`drop` 등으로 이미 사라진 컬럼일 수 있습니다.
- **해결:** `df.columns`로 지금 실제로 있는 컬럼명을 먼저 확인하세요.
```python
print(df.columns.tolist())  # 실제 컬럼명 확인 (대소문자·띄어쓰기 오타 주의)
# get_dummies나 drop 이후라면, 바뀐 뒤의 컬럼명을 그대로 써야 합니다.
```

### 🚨 ValueError: Shapes (None, 1) and (None, 3) are incompatible
- **원인:** 다중분류인데 출력층 노드를 1개로 만들었거나, 반대로 이진분류인데 여러 개로 만들었을 때, 또는 `loss`를 문제 유형과 다르게 설정했을 때 발생합니다.
- **해결:** 클래스 개수(N)에 맞춰 출력층 노드 수와 `loss`를 함께 맞추세요.
```python
# 이진분류: 출력 노드 1개 + sigmoid + binary_crossentropy
model.add(Dense(1, activation='sigmoid'))
model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

# 다중분류(클래스 3개인 경우): 출력 노드 3개 + softmax + sparse_categorical_crossentropy
model.add(Dense(3, activation='softmax'))
model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
```
